# Eleven PIT daily-signal backtests

This is the active report. It imports `us_equities_signal_research`; it does not import `retired/legacy_runtime` or `signal-laboratory-main`.


In [ ]:
from pathlib import Path
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd
import us_equities_datahandler as dh
import us_equities_signal_research as signals

dh.configure(
    crsp_ranked_daily='/datahandler/data/derived/crsp_ranked_daily',
    compustat_annual='/datahandler/data/derived/compustat/annual.parquet',
    ccm_links='/datahandler/data/derived/ccm/links.parquet',
)

START, END = '2000-01-01', '2025-12-31'
RANK_LTES = [500, 1000, 2000]
FREQUENCIES = ['daily', 'weekly', 'monthly']
WEIGHT_METHODS = ['zscore', 'rank', 'quintile']
SIC_LEVELS = [None, 1]  # None: global score; 1: 11 supplied SIC divisions
RESULT_ROOT = Path('/cache/backtests')


In [ ]:
# Run once to build all eleven raw daily panels and the complete portfolio grid
# for the three requested dynamic size universes. Subsequent runs reuse caches.
daily_parts, metric_parts = [], []
for rank_lte in RANK_LTES:
    universe = dh.universe.market_cap(start=START, end=END, rank_lte=rank_lte, lag=1)
    result_dir = RESULT_ROOT / f'top{rank_lte}_{START[:4]}_{END[:4]}'
    daily_one, metrics_one = signals.run_backtest(
        data=dh, universe=universe, output=result_dir,
        frequencies=FREQUENCIES, methods=WEIGHT_METHODS, sic_levels=SIC_LEVELS,
    )
    daily_one['rank_lte'] = rank_lte
    metrics_one['rank_lte'] = rank_lte
    daily_parts.append(daily_one)
    metric_parts.append(metrics_one)
daily = pd.concat(daily_parts, ignore_index=True)
metrics = pd.concat(metric_parts, ignore_index=True)
display(metrics.sort_values(['rank_lte', 'signal', 'Sharpe'], ascending=[True, True, False]).reset_index(drop=True))


In [ ]:
# Or load completed runs without recomputing them.
daily_parts, metric_parts = [], []
for rank_lte in RANK_LTES:
    result_dir = RESULT_ROOT / f'top{rank_lte}_{START[:4]}_{END[:4]}'
    daily_one = pd.read_parquet(result_dir / 'daily_ls_returns.parquet')
    metrics_one = pd.read_csv(result_dir / 'performance_metrics.csv')
    daily_one['rank_lte'] = rank_lte
    metrics_one['rank_lte'] = rank_lte
    daily_parts.append(daily_one)
    metric_parts.append(metrics_one)
daily = pd.concat(daily_parts, ignore_index=True)
metrics = pd.concat(metric_parts, ignore_index=True)

RANK_LTES = [500]
FREQUENCIES = ['daily']
WEIGHT_METHODS = ['zscore']
SIC_LEVELS = [None]

def sic_mask(frame, level):
    return frame.sic_level.isna() if level is None else frame.sic_level.eq(level)

selection = metrics.rank_lte.isin(RANK_LTES) & metrics.rebalance_frequency.isin(FREQUENCIES) & metrics.weight_method.isin(WEIGHT_METHODS)
sic_selection = pd.Series(False, index=metrics.index)
for level in SIC_LEVELS:
    sic_selection |= sic_mask(metrics, level)
display(metrics.loc[selection & sic_selection].sort_values('Sharpe', ascending=False).reset_index(drop=True))


In [ ]:
# Eleven subplots; each selected Cartesian-product configuration becomes one curve.
signals_in_run = sorted(daily.signal.unique())
fig, axes = plt.subplots(4, 3, figsize=(18, 17), sharex=True)
axes = axes.ravel()
for ax, signal in zip(axes, signals_in_run):
    for rank_lte, frequency, method, sic_level in product(RANK_LTES, FREQUENCIES, WEIGHT_METHODS, SIC_LEVELS):
        mask = (
            daily.signal.eq(signal)
            & daily.rank_lte.eq(rank_lte)
            & daily.rebalance_frequency.eq(frequency)
            & daily.weight_method.eq(method)
            & sic_mask(daily, sic_level)
        )
        curve = daily.loc[mask].sort_values('date')
        if not curve.empty:
            sic_name = 'global' if sic_level is None else f'SIC {sic_level}'
            ax.plot(curve.date, 100 * curve.cum_return, lw=.8, alpha=.85,
                    label=f'top {rank_lte} | {frequency} | {method} | {sic_name}')
    ax.axhline(0, color='black', lw=.5)
    ax.set_title(signal, fontsize=9)
    ax.grid(alpha=.2)
for ax in axes[len(signals_in_run):]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Universe | rebalance | weights | score treatment',
           loc='lower center', ncol=3, fontsize=7)
fig.suptitle('Cumulative simple excess return by signal and configuration', y=.995)
fig.tight_layout(rect=(0, .08, 1, .98))
plt.show()


Raw panels are daily `date × permno` caches. The score step applies the dynamic universe mask, global 1% winsorization, optional SIC-division scoring, and global net-zero / gross-one normalization. The return step shifts target weights two CRSP sessions: information known after close on *t* forms at *t+1* close and earns the *t+1 → t+2* close-to-close return.
